# NumCompute Quickstart

A short tour of each NumCompute module, with runnable examples.

**To run it:**
1. From the project root, run `uv sync` (see the README).
2. Open this notebook and choose the `.venv` Python as the kernel. If VS Code asks to install `ipykernel`, accept.
3. Run all cells from top to bottom. Paths assume the notebook runs from the `demo/` folder.

In [5]:
from pathlib import Path

import numpy as np

DATA_DIR = Path("..") / "tests" / "test_csv_files"

## 1. Loading data (`io`)

`load_csv` reads a CSV into a 2D float array. Missing values become `NaN`.

In [6]:
from numcompute.io import load_csv

X = load_csv(DATA_DIR / "missing_data_multiple.csv")
print(X.shape)
X

ModuleNotFoundError: No module named 'numcompute'

## 2. Preprocessing (`preprocessing`)

Every transformer has `fit(X)`, `transform(X)` and `fit_transform(X)`. Calling `transform` before `fit` raises `NotFittedError`.

### Filling missing values: `SimpleImputer`

In [7]:
from numcompute.preprocessing import SimpleImputer

X_filled = SimpleImputer(fill_value=0.0).fit_transform(X)
X_filled

ModuleNotFoundError: No module named 'numcompute'

### Scaling: `StandardScaler` and `MinMaxScaler`

`StandardScaler` gives each column mean 0 and standard deviation 1. `MinMaxScaler` maps each column into `feature_range`.

In [ ]:
from numcompute.preprocessing import MinMaxScaler, StandardScaler

X_std = StandardScaler().fit_transform(X_filled)
print("column means:", X_std.mean(axis=0).round(6) + 0.0)
print("column stds: ", X_std.std(axis=0).round(6))

X_minmax = MinMaxScaler(feature_range=(-1, 1)).fit_transform(X_filled)
X_minmax

### Learn on training data, apply to new data

`fit` learns from one dataset; `transform` reuses what was learned on any new data with the same columns.

In [ ]:
scaler = StandardScaler().fit(X_filled)

new_data = np.array([[5.0, 5.0, 5.0]])
scaler.transform(new_data)

### Categorical data: `OneHotEncoder`

Each column becomes one 0/1 column per category (sorted). Unknown categories raise an error unless `handle_unknown="ignore"`.

In [ ]:
from numcompute.preprocessing import OneHotEncoder

colours = [["red", "S"], ["blue", "L"], ["red", "L"]]

encoder = OneHotEncoder().fit(colours)
print(encoder.categories_)
encoder.transform(colours)

In [ ]:
OneHotEncoder(handle_unknown="ignore").fit(colours).transform([["green", "S"]])

## 3. Numerical gradients (`optim`)

`grad` and `jacobian` estimate derivatives with finite differences. `method="central"` (default) is more accurate than `"forward"`.

The gradient of $f(x) = \sum x_i^2$ is $2x$:

In [ ]:
from numcompute.optim import grad, jacobian


def sum_of_squares(x):
    return np.sum(x**2)


x = np.array([1.0, 2.0, 3.0])
print("central:", grad(sum_of_squares, x))
print("forward:", grad(sum_of_squares, x, method="forward"))

For a function with several outputs, `jacobian` returns shape `(outputs, inputs)`. For $F(x) = [x_0 x_1,\ \sin x_0]$ the exact answer is $\begin{bmatrix} x_1 & x_0 \\ \cos x_0 & 0 \end{bmatrix}$:

In [ ]:
def vector_func(x):
    return np.array([x[0] * x[1], np.sin(x[0])])


jacobian(vector_func, [1.0, 2.0])

## 4. Sort and Search (`sort_search`)

`sort_search` provides stable and multi-key sorting, efficient top-k selection, manual Quickselect, and binary search.

In [ ]:
from numcompute.sort_search import (
    stable_sort,
    multi_key_sort,
    topk,
    quickselect,
    binary_search,
)

In [ ]:
### Stable sorting

values = np.array([5, 2, 4, 2, 1])

stable_sort(values

### Multi-key sorting

`multi_key_sort` sorts the rows of a 2D array by using its column indices in order of priority. The first key is always the sorting key.

In [ ]:
data = np.array([
    [2, 30],
    [1, 20],
    [2, 10],
    [1, 10],
])

multi_key_sort(data, [0,1])

### Top-k selection

`topk` uses partial sorting to efficiently return the largest of the smallest `k` values

In [ ]:
values = np.array([4, 11, 2, 19, 7])

print("largest 3 values:", topk(values, 3, largest=True, return_indices=False))
print("largest 3 indices:", topk(values, 3, largest=True, return_indices=True))
print("smallest 2 values:", topk(values, 2, largest=False, return_indices=False))

### Quickselect

`quickselect` finds the k-th smallest value without fully sorting the array. `k` uses 1-based indexing. The algorithm modifies its input array in place, so copy is used below.

In [ ]:
values = np.array([6, 1, 8, 0, 5, 3])

third_smallest = quickselect(values.copy(), 3)

print("3rd smallest is:", third_smallest)

### Binary search

`binary_search` searches a sorted array and returns the index and a boolean indicating whether the value was present in the array. If the value was absent, i.e. the Boolean was False, then that index is where it is inserted.

In [ ]:
sorted_values = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9])

print("Search for 5:" binary_search(sorted_values, 5))
prtin("Search for 10:" binary_search(sorted_values, 10))

## 8. Pipelines (`pipeline`)

`Pipeline` chains multiple preprocessing steps such that the output of a transformer is automatically passed as the input of the next.

In [ ]:
from numcompute.pipeline import Pipeline
from numcompute.preprocessing import StandardScaler, MinMaxScaler

### Chaining preprocessing steps

`fit_transform` fits each transformer in a sequence, returnin only the final transformed data.

In [8]:
X_pipe = np.array([
    [1.0, 10.0],
    [2.0, 20.0],
    [3.0, 30.0],
])

pipe = Pipeline([
    ("scale", StandardScaler()),
    ("minmax", MinMaxScaler()),
])

X_transformed = pipe.fit_transform(X_pipe)

X_transformed
    

_IncompleteInputError: incomplete input (1811495802.py, line 7)

### Applying fitted pipeline to new data

After fitting is completed, `transform` reuses parameters learned from original data, instead of repeat the process by fitting the preprocessing steps again.

In [ ]:
new_data = np.array([
    [1.5, 15.0],
    [2.5, 25.0],
])

pipe.transform(new_data)

A pipeline could also end up with an estimator that provides `fit` and `predict`. As NumCompute does not implement a machine learning model, the example above only focuses on chaining preprocessing transformers.

## 10. Other modules

*Module owners: add a section here for your module (`rank`, `stats`, `metrics`, `pipeline`), following the pattern above: a short heading, one sentence, then a small runnable example.*